In [11]:
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
import re
import ftfy
import html
pd.set_option('display.max_colwidth', None)

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
n_nan_source = df['source'].isna().sum()
n_empty_soruce = df['source'].astype(str).str.strip().eq('').sum()
n_placeholders_source = df['source'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_source}")
print(f"Number of empty rows: {n_empty_soruce}")
print(f"Number of placeholders (\\N): {n_placeholders_source}")
df['source'] = df['source'].replace('\\N', 'Unknown')
source_counts = df['source'].value_counts()
selected_sources = source_counts[source_counts >= 50].index.to_list()
selected_sources.remove('Unknown')
print(f"Relevant Sources:\n{selected_sources}")
coverage = source_counts[selected_sources].sum() / len(df)
print(f"Number of selected sources: {len(selected_sources)}")
print(f"Percentage of selected sources: {coverage:.2%}")

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 294
Relevant Sources:
['Yahoo', 'Reuters', 'BBC', 'New', 'Washington', 'RedNova', 'Boston', 'CNN', 'CNET', 'Topix.Net', 'Guardian', 'Motley', 'Register', 'International', 'Forbes', 'Time', 'ABC', 'InfoWorld', 'San', 'Wired', 'Xinhua', 'Computerworld', 'News', 'CSMonitor', 'PCWorld', 'Bloomberg', 'Seattle', 'Ananova', 'Syfy.com', 'Voice', 'USA', 'Independent', 'Scotsman', 'CBS', 'Rediff', 'Times', 'Channel', 'CBC', 'Newsday', 'Newsweek', 'Houston', 'Australian', 'Daily', 'Telegraph.co.uk', 'ESPN', 'Canada.com', 'BCC', 'Sports', 'Search', 'Chicago', 'Turkish', 'CNN/SI', 'MSNBC', 'London', 'National', 'Financial', 'Toronto', 'Indianapolis', 'Melbourne', 'Christian', 'Detroit', 'ZDNet.com', 'CTV', 'PC', 'ic', 'NEWS.com.au', 'RTE', 'Scotland', 'Hindustan', 'NPR', 'Al-Jazeera', 'Information', 'IPS', 'TechNewsWorld', 'News24', 'sportinglife.com', 'Arizona', 'Age', 'Taipei', 'Radio', 'Gulf', 'Sun-Sentinel.com', 'Indian'

### *Title* feature inspection

In [4]:
n_nan_title = df['title'].isna().sum()
n_empty_title = df['title'].astype(str).str.strip().eq('').sum()
n_placeholders_title = df['title'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of empty rows: {n_empty_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print("Titles Sample:")
print(df['title'].sample(20))

Number of NaN rows: 1
Number of empty rows: 2
Number of placeholders (\N): 0
Titles Sample:
Id
30499                        Fool on the Street: Rock-Solid Wells Fargo
61096                                     A Huge Investment Opportunity
24330           Crude oil prices fall slightly on better crude supplies
60245    Japan suspects North Korea missile moves were only an exercise
79825                Cronenberg astonishes with <I>Eastern Promises</I>
2513                                  Officers widen murder questioning
16004         High & Low Finance: Banks Plead They Canât Follow Rules
23786                          Lawmakers Question Merck, FDA Over Vioxx
4065               Lycos Europe Confronts Strong Resistance In Spam War
19668                         Sudan translator &#39;seized in Iraq&#39;
53639                      U.S.-Style Entertainment Rocks Olympics (AP)
35550                        Navy Says Kerry's Service Awards OK'd (AP)
3865                                     

### *Article* feature inspection

In [5]:
n_nan_article = df['article'].isna().sum()
n_empty_article = df['article'].astype(str).str.strip().eq('').sum()
n_placeholders_article = df['article'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of empty rows: {n_empty_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print("Articles Sample")
print(df['article'].sample(10))

Number of NaN rows: 1
Number of empty rows: 7
Number of placeholders (\N): 1874
Articles Sample
Id
5511                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 A civil court has ruled that a popular tax-avoidance scheme known as Son of Boss was abusive and any deductions claimed for it were invalid.
71967                                                                                                                              

### *PageRank* feature inspection

In [6]:
n_nan_pr = df['page_rank'].isna().sum()
n_empty_pr = df['page_rank'].astype(str).str.strip().eq('').sum()
n_placeholders_pr = df['page_rank'].astype(str).str.strip().eq('\\N').sum()
rank_5=np.array([df['page_rank'].values==5]).sum()
print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")
print(df['page_rank'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number of articles with PageRank 5: 73891
Id
45744    5
77735    5
60048    5
45115    5
71724    5
8808     5
79269    5
13549    5
17539    5
74628    5
Name: page_rank, dtype: int64


### *Timestamp* feature inspection 

In [7]:
n_nan_time = df['timestamp'].isna().sum()
n_empty_time = df['timestamp'].astype(str).str.strip().eq('').sum()
n_placeholders_time = df['timestamp'].astype(str).str.strip().eq('\\N').sum()
n_uslesess_time=np.array([df['timestamp'].values=="0000-00-00 00:00:00"]).sum()
print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number invalid dates (0000-00-00 00:00:00): 27750
Id
63310    2007-06-04 05:35:57
63568    0000-00-00 00:00:00
35406    2006-09-06 21:42:33
1256     2008-02-05 14:14:04
54732    2004-09-08 09:07:17
37454    2006-09-26 15:00:30
25813    0000-00-00 00:00:00
8281     2007-06-01 03:20:32
32666    2007-12-14 11:55:53
70278    0000-00-00 00:00:00
Name: timestamp, dtype: object


### *Timestamp* feature processing

In [8]:
def process_timestamp(df):
    df = df.copy()
    df['dt_obj'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    df['has_date'] = df['dt_obj'].notna().astype(int)
    df['year'] = df['dt_obj'].dt.year.fillna(-1).astype(int)
    df['month'] = df['dt_obj'].dt.month.fillna(-1).astype(int)
    df['day_of_week'] = df['dt_obj'].dt.dayofweek.fillna(-1).astype(int)
    df['hour'] = df['dt_obj'].dt.hour.fillna(-1).astype(int)

    df = df.drop(columns=['timestamp', 'dt_obj'])
    
    return df

df = process_timestamp(df)
new_cols = ['has_date', 'year', 'month', 'day_of_week', 'hour']
print(f"New columns added: {new_cols} , Column removed: ['timestamp']")
print("Test new timestamp features:\n")
print(df[new_cols].sample(10))

New columns added: ['has_date', 'year', 'month', 'day_of_week', 'hour'] , Column removed: ['timestamp']
Test new timestamp features:

       has_date  year  month  day_of_week  hour
Id                                             
24466         1  2006      5            6    17
61201         0    -1     -1           -1    -1
44890         1  2008      1            2    23
15204         1  2007      2            6    21
12391         0    -1     -1           -1    -1
52133         1  2007      7            6     8
77822         0    -1     -1           -1    -1
13489         1  2004     12            5     9
13317         1  2008      1            2    15
36783         1  2007      7            5    16


### *Title* feature stemming

In [ ]:
stop_words = set(stopwords.words('english'))
if 'us' in stop_words:
    stop_words.remove('us') 

stemmer = SnowballStemmer("english")

def clean_title(text):
    if pd.isna(text) or text == "": return ""
    text = str(text)
    text = html.unescape(text)
    text = ftfy.fix_text(text)
    text = text.lower()
    
    text = text.replace('.', '') 
    
    text = re.sub(r'(?:\\n|\s)*\(.*?\)\W*$', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s+(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z]', ' ', text)

    words = text.split()
    
    meaningful_words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) >= 2]
    
    return " ".join(meaningful_words)


def test_simple(titles_series, cleaner_func, n):
    sample = titles_series.sample(n)
    print(f"Test on {n} random titles\n")
    for _, text in sample.items():
        cleaned = cleaner_func(text)
        print(f"Original text: {text}")
        print(f"Processed text: {cleaned}")
        print("-" * 50)

test_simple(df['title'], clean_title, n=10)

Test on 100 random titles

Original text: Russia, U.N. Voice Fallujah Concerns (AP)
Processed text: russia un voic fallujah concern
--------------------------------------------------
Original text: N.Korea calls Syria nuclear ties report "conspiracy"
Processed text: nkorea call syria nuclear tie report conspiraci
--------------------------------------------------
Original text: Crisis talks as police thwart major double attack on London \
    (AFP)\

Processed text: crisi talk polic thwart major doubl attack london
--------------------------------------------------
Original text: Feds to probe Comcast's BitTorrent busting
Processed text: fed probe comcast bittorr bust
--------------------------------------------------
Original text: Agent Orange appeal in US court
Processed text: agent orang appeal us court
--------------------------------------------------
Original text: Anorexic Girls Bond on Web to Dismay of Doctors
Processed text: anorex girl bond web dismay doctor
----------------

### *Article* feature stemming

In [ ]:
def clean_article(text):
    if pd.isna(text) or text == "" or str(text).strip() == "\\N": 
        return ""
    text = str(text)

    text = text[:1000] 
    text = html.unescape(text)
    text = re.sub(r'http[s]?://\S+', ' ', text)
    text = re.sub(r'\b[a-z0-9]+\.(com|net|org|gov)\b', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    trash_pattern = r'\b(src|href|alt|width|height|align|border|style|sig|valign|hspace|vspace)\b'
    text = re.sub(trash_pattern, ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b[a-z0-9]*[0-9]+[a-z0-9]*\b', ' ', text) 
    text = re.sub(r'\b[bcdfghjklmnpqrstvwxyz]{4,}\b', ' ', text) 

    text = ftfy.fix_text(text)
    text = text.lower()
    text = text.strip()
    
    text = re.sub(r'^\s*[a-z][\w\s,\.\(\)]{0,50}\s*--\s*', '', text)
    text = re.sub(r'^\s*[a-z][^\.\?!]{2,50}\s+[-–—]\s+', '', text)
    
    agencies_pattern = r'(?i)^\s*.*?\b(reuters|afp|ap|upi|bloomberg|bbc|cnn|blog)\b.*?\s*[-:–—]\s*'
    text = re.sub(agencies_pattern, '', text)

    text = re.sub(r'(?i)^by\s+[a-z\s\.,]+\s{2,}', '', text)
    text = text.replace('.', '')

    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s+(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z]', ' ', text)

    words = text.split()
    
    meaningful_words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) >= 2]
    
    return " ".join(meaningful_words)

def test_simple_article(article_series, cleaner_func, n):
    sample = article_series.sample(n)
    print(f"Test on {n} random articles\n")
    for _, text in sample.items():
        print(f"Original text (First 200 char): {str(text)}") 
        print(f"Processed text: {cleaner_func(text)}")
        print("-" * 50)

test_simple_article(df['article'], clean_article, n=10)

Test on 100 random articles

Original text (First 200 char): Several thousand Israeli Arabs have marched through the streets of Nazareth in a symbolic funeral for Palestinian president Yasser Arafat.
Processed text: sever thousand isra arab march street nazareth symbol funer palestinian presid yasser arafat
--------------------------------------------------
Original text (First 200 char): AP - A year after the nation's worst blackout, federal regulators issued a scathing review Wednesday of the electricity industry's voluntary efforts to make their power grids more reliable.
Processed text: year nation worst blackout feder regul issu scath review wednesday electr industri voluntari effort make power grid reliabl
--------------------------------------------------
Original text (First 200 char):  KABUL, June 6 -- Afghanistan's recent spate of violence claimed the lives of two more NATO soldiers Wednesday, while the death toll in June among insurgents rose to 200. 
Processed text: afghani

### *Title + Article* features merge

In [19]:
print(f"Starting shape: {df.shape}")
print(f"Starting columns: {df.columns.tolist()}")
df['title_clean'] = df['title'].apply(clean_title)
df['article_clean'] = df['article'].apply(clean_article)
df['text_combined'] = (df['title_clean'] + " " + df['title_clean'] + " " + df['article_clean']).str.strip()

n_empty = (df['text_combined'] == "").sum()
print(f"Removing {n_empty} rows with empty text")
df = df[df['text_combined'] != ""]
df = df.drop(columns=['title', 'article', 'title_clean', 'article_clean'])

print(f"Final shape: {df.shape}")
print(f"Actual columns: {df.columns.tolist()}")
print("Example of title + article combined:")
print(df['text_combined'].iloc[0])

Starting shape: (79997, 10)
Starting columns: ['source', 'title', 'article', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'hour']
Removing 3 rows with empty text
Final shape: (79994, 9)
Actual columns: ['source', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'hour', 'text_combined']
Example of title + article combined:
opec boost nigeria oil revenu bpd opec boost nigeria oil revenu bpd organis petroleum export countri opec hike offici output one million barrel per day effect novemb nigeria get barrel per day per cent new quota


### Encoding categorical features

In [20]:
df['source'] = np.where(df['source'].isin(selected_sources), df['source'], 'Other')
categorical_cols = ['source', 'year', 'month', 'day_of_week', 'hour']

print(f"Shape before encoding of categorical features: {df.shape}")

df = pd.get_dummies(
    df, 
    columns=categorical_cols, 
    prefix=categorical_cols, 
    prefix_sep='_', 
    dtype=int
)

print(f"Shape after encoding of categorical features: {df.shape}")
new_cols = [c for c in df.columns if c not in ['label', 'text_combined', 'page_rank', 'has_date']]
print(f"New encoded columns sample: {new_cols[::15]}")

Shape before encoding of categorical features: (79994, 9)
Shape after encoding of categorical features: (79994, 155)
New encoded columns sample: ['source_ABC', 'source_CNN', 'source_Financial', 'source_Ireland', 'source_Newsday', 'source_Search', 'source_USA', 'year_2008', 'day_of_week_1', 'hour_8', 'hour_23']


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack 

target_col = 'label' 
custom_stop_words = [
    'said','say' ,'says','report', 'reported','according', 'today', 'yesterday', 'tomorrow', 'week', 'month', 'day','year', 'years', 'time']
my_stop_words = list(ENGLISH_STOP_WORDS) + custom_stop_words

y = df[target_col]
X = df.drop(columns=[target_col])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.8,
    stop_words=my_stop_words,
    sublinear_tf=True,
    norm='l2'
)

X_train_vectorized = tfidf.fit_transform(X_train['text_combined'])
X_test_vectorized = tfidf.transform(X_test['text_combined'])

scaler = MinMaxScaler()
X_train_pr_scaled = scaler.fit_transform(X_train[['page_rank']])
X_test_pr_scaled = scaler.transform(X_test[['page_rank']])

drop_cols = ['text_combined', 'page_rank']
X_train_base = X_train.drop(columns=drop_cols)
X_test_base = X_test.drop(columns=drop_cols)

X_train_final = hstack([X_train_vectorized, X_train_pr_scaled, X_train_base])
X_test_final = hstack([X_test_vectorized, X_test_pr_scaled, X_test_base])

print(f"Matrix Train: {X_train_final.shape}")
print(f"Matrix Test:  {X_test_final.shape}")

Train set: (63995, 154)
Test set:  (15999, 154)
Matrix Train: (63995, 10153)
Matrix Test:  (15999, 10153)


In [22]:
# CELLA 12: Training con XGBoost (L'artiglieria pesante)

from xgboost import XGBClassifier
from random import randint
from sklearn.metrics import classification_report, f1_score
import time

# 1. PREPARAZIONE XGBOOST
# XGBoost è molto potente ma richiede label numeriche partendo da 0 (lo abbiamo già)
# Usiamo i parametri "Gold Standard" per text classification
xgb_model = XGBClassifier(
    n_estimators=1000,          # Tanti alberi
    learning_rate=0.05,         # Impara lentamente (più preciso)
    max_depth=randint(3, 8),                # Profondità media
    min_child_weight=1,
    subsample=0.8,              # Usa l'80% dei dati per ogni albero (evita overfitting)
    colsample_bytree=0.8,       # Usa l'80% delle feature per ogni albero
    objective='multi:softprob', # Classificazione multiclasse
    num_class=len(y.unique()),  # Numero di classi
    n_jobs=-1,                  # Parallelo
    random_state=42,
    early_stopping_rounds=50    # Se non migliora per 50 round, fermati (risparmia tempo)
)

# 2. TRAINING CON EARLY STOPPING
# Dobbiamo passargli un evaluation set per capire quando fermarsi
# Usiamo il test set come eval (in produzione si userebbe un validation set separato, ma qui va bene per ottimizzare)
print("Avvio training XGBoost...")
start_time = time.time()

xgb_model.fit(
    X_train_final, y_train,
    eval_set=[(X_train_final, y_train), (X_test_final, y_test)],
    verbose=100  # Stampa ogni 100 iterazioni
)

end_time = time.time()
print(f"\nTraining completato in {(end_time - start_time)/60:.1f} minuti.")

# 3. VALUTAZIONE
print("\n--- RISULTATI XGBOOST ---")
y_pred_xgb = xgb_model.predict(X_test_final)

print(classification_report(y_test, y_pred_xgb))
print(f"MACRO F1 SCORE (XGBoost): {f1_score(y_test, y_pred_xgb, average='macro'):.4f}")

Avvio training XGBoost...
[0]	validation_0-mlogloss:1.87402	validation_1-mlogloss:1.87387
[100]	validation_0-mlogloss:1.13511	validation_1-mlogloss:1.15065
[200]	validation_0-mlogloss:0.99914	validation_1-mlogloss:1.03369
[300]	validation_0-mlogloss:0.92497	validation_1-mlogloss:0.97602
[400]	validation_0-mlogloss:0.87300	validation_1-mlogloss:0.93937
[500]	validation_0-mlogloss:0.83325	validation_1-mlogloss:0.91314
[600]	validation_0-mlogloss:0.80074	validation_1-mlogloss:0.89372
[700]	validation_0-mlogloss:0.77360	validation_1-mlogloss:0.87854
[800]	validation_0-mlogloss:0.74976	validation_1-mlogloss:0.86607
[900]	validation_0-mlogloss:0.72855	validation_1-mlogloss:0.85571
[999]	validation_0-mlogloss:0.70991	validation_1-mlogloss:0.84767

Training completato in 18.6 minuti.

--- RISULTATI XGBOOST ---
              precision    recall  f1-score   support

           0       0.66      0.81      0.73      4708
           1       0.75      0.78      0.76      2118
           2       0.84